# 1. Downloading the OASIS-1 raw MRI dataset

This notebook documents the acquisition of the **OASIS-1 raw MRI archives**. It separates downloading, extraction, and cleanup so that each operation can be inspected and repeated independently.

**Before running:** Review the [OASIS data-use terms](https://sites.wustl.edu/oasisbrains/request-access/) and ensure that you have any required access permissions. Run this notebook from the project environment (with the `Neuroimaging` package importable). The notebook is intended to live at `<project root>/notebooks/1_download_data.ipynb`.

The original archives are downloaded into `data/raw/.archives/`; extracted files go into `data/raw/`. Successful extractions are tracked in `data/raw/.extracted/`.

## 0. Configuration

Choose the discs to process. By default, all twelve are selected. Paths are anchored to the project root rather than the notebook's current working directory. The next cell assumes the notebook kernel starts either in the project root or in its `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
if (working_directory / "Neuroimaging").is_dir():
    project_root = working_directory
elif (working_directory.parent / "Neuroimaging").is_dir():
    project_root = working_directory.parent
else:
    raise RuntimeError("Start the notebook in the project root or notebooks/ directory.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Neuroimaging import BASE_URL, ARCHIVE_TEMPLATE, download, extract

discs = list(range(1, 13))  # Example: [1, 2, 3]
if not discs or any(disc not in range(1, 13) for disc in discs):
    raise ValueError("Choose disc numbers between 1 and 12.")
discs = list(dict.fromkeys(discs))  # Remove duplicates, preserving order.

destination = project_root / "data" / "raw"
archives = destination / ".archives"
markers = destination / ".extracted"

archives.mkdir(parents=True, exist_ok=True)
markers.mkdir(parents=True, exist_ok=True)

print(f"Project: {project_root}")
print(f"Selected discs: {discs}")
print(f"Raw data: {destination}")

## 1. Download archives

Download the selected `.tar.gz` files into `.archives/`. A disc already marked as successfully extracted is skipped. The `download()` helper must handle incomplete downloads safely (for example, by resuming or replacing partial files); check its implementation before relying on that behavior. **Do not delete the archives yet.**

In [ ]:
for disc in discs:
    filename = ARCHIVE_TEMPLATE.format(disc=disc)
    archive = archives / filename
    marker = markers / f"disc{disc}.done"

    if marker.exists():
        print(f"Disc {disc}: already extracted; skipping download")
        continue

    print(f"Downloading disc {disc}: {filename}", flush=True)
    download(f"{BASE_URL}/{filename}", archive)

print("Download stage finished.")

## 2. Extract archives

Extract each available archive into `data/raw/`. The `extract()` helper is responsible for safe extraction and for writing the `.done` marker **only after successful completion**. If extraction fails, the archive is retained so that the operation can be retried. This stage can be rerun without repeating completed extractions.

In [ ]:
for disc in discs:
    filename = ARCHIVE_TEMPLATE.format(disc=disc)
    archive = archives / filename
    marker = markers / f"disc{disc}.done"

    if marker.exists():
        print(f"Disc {disc}: already extracted; skipping")
        continue
    if not archive.is_file():
        raise FileNotFoundError(f"Missing archive for disc {disc}: {archive}")

    print(f"Extracting disc {disc}: {filename}", flush=True)
    extract(archive, destination, marker)

print("Extraction stage finished.")

## 3. Cleanup (optional)

After verifying the extraction, you can remove the compressed archives to save disk space. This operation **does not remove extracted MRI files**. Only archives whose discs have a completion marker are eligible for deletion. Set `delete_archives = True` explicitly to perform cleanup. Keep the archives if you need to preserve the exact downloaded inputs for auditing or offline re-extraction.

**Reproducibility note:** Completion markers are a convenience, not a substitute for verifying file integrity or documenting dataset versions/checksums. The notebook depends on the behavior of the `Neuroimaging.download` and `Neuroimaging.extract` helpers.

In [ ]:
delete_archives = False  # Change to True only after verifying extracted data.

for disc in discs:
    archive = archives / ARCHIVE_TEMPLATE.format(disc=disc)
    marker = markers / f"disc{disc}.done"

    if not archive.exists():
        continue
    if not marker.exists():
        print(f"Keeping disc {disc}: extraction not confirmed")
        continue

    if delete_archives:
        archive.unlink()
        print(f"Deleted {archive.name}")
    else:
        print(f"Keeping {archive.name}")

print("Cleanup stage finished.")

## Reproducibility record

Record the date of acquisition, OASIS access conditions, selected discs, software/environment version, and any errors or retries in the accompanying project documentation. For stronger verification, record SHA-256 checksums of downloaded archives **before** optional cleanup. Do not commit large MRI data files or restricted data to a public repository.